# 2.2 Using the OpenAI SDK

### I. OpenAI Chat SDK Core Features
OpenAI provides official and community-maintained SDKs in multiple languages for quickly integrating ChatGPT functionality into applications. Core features include:
1. **Multi-language support**: Covers Python, Java, .NET, and other mainstream development languages
2. **Feature completeness**: Supports chat completions, function calling, streaming responses, and other core API features
3. **Developer-friendly**: Provides sync/async interfaces, pre-built request objects, automatic error retry mechanisms

Here are the core features and parameter settings of the OpenAI standard Python SDK, compatible with mainstream LLM calling scenarios (such as ChatGPT, GPT-4, etc.), with parameter logic extensible to other OpenAI API-compatible models:

---

### I. SDK Core Features and Compatibility Design
1. **Unified Interface Specification**
   The OpenAI SDK provides model calls through the standardized `ChatCompletion.create()` method, compatible with all services following the OpenAI interface specification (such as various third-party LLM providers).
   ```python
   from openai import OpenAI
   client = OpenAI(api_key="YOUR_KEY", base_url="https://your-model-api-url")  # Key compatibility parameter
   ```

2. **Cross-model Parameter Mapping**
   Core parameters (such as `temperature`, `max_tokens`) have consistent semantics across different models; only the value ranges need adjustment to adapt to different model characteristics.

---

### II. Key Parameter Configuration Guide
| Parameter          | Function                                                              | Recommended Range | Use Case                          |
|--------------------|-----------------------------------------------------------------------|-------------------|-----------------------------------|
| **model**          | Specifies model version (e.g., `gpt-3.5-turbo`)                      | Model support list | Controls model capability and cost |
| **temperature**    | Output randomness: lower values are more deterministic, higher more creative | 0.2-1.0           | Factual Q&A (0.2), Creative (0.8) |
| **max_tokens**     | Maximum generated text length (including input+output)                | 50-4096           | Controls response length and API cost |
| **top_p**          | Nucleus sampling threshold: samples only from candidates whose cumulative probability meets the threshold | 0.7-0.95          | Balances diversity and relevance   |
| **stream**         | Enables streaming response (token-by-token output)                    | True/False        | Improves interaction experience    |
| **presence_penalty** | Suppresses repeated content (positive suppresses, negative allows)  | -2.0-2.0          | Avoids redundant answers           |

#### Example: Multi-model Compatible Calls
```python
# Calling GPT-3.5 (official model)
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Explain quantum entanglement"}],
    temperature=0.5,
    max_tokens=500
)

# Calling another model
response = client.chat.completions.create(
    model="qwen-max",
    messages=[{"role": "user", "content": "Write a short poem"}],
    temperature=0.7,
    top_p=0.9
)
```

---

### III. Best Practices and Notes
1. **Parameter Tuning Strategy**
   - Combine `temperature`+`top_p` for precise control (e.g., `temperature=0.8` + `top_p=0.9` balances creativity and logic)
   - Use `max_tokens` to limit output length, avoiding exceeding the model's context limit (e.g., GPT-3.5's 4096 tokens)

2. **Error Handling**
   ```python
   try:
       response = client.chat.completions.create(...)
   except openai.APIError as e:
       print(f"API Error: {e}")
   except openai.RateLimitError:
       print("Rate limit exceeded, please retry later")
   ```

Here we use the IO Solutions API as an example; other providers are similar:
https://api.intelligence.io.solutions/api/v1

<img src="img/huoshan.png" width="600px" />

In [ ]:
!pip install --upgrade "openai>=1.0"

In [ ]:
ARK_API_KEY = "YOUR_IO_API_KEY"

In [ ]:
import os
from openai import OpenAI

# Make sure you have stored your API Key
# Initialize the OpenAI client
client = OpenAI(
    # Default path; you can configure based on your region
    base_url="https://api.intelligence.io.solutions/api/v1",
    api_key="YOUR_IO_API_KEY",
)

In [ ]:
# Non-streaming:
completion = client.chat.completions.create(
    model="moonshotai/Kimi-K2.6",
    messages=[
        {"role": "user", "content": "What are the common drone simulation systems?"},
    ],
    temperature=0.1,
)
print(completion.choices[0].message.content)

## Using Roles

Here is a comparison table of message roles in the OpenAI SDK, showing the core functions, typical uses, and examples of different roles:

| Role        | Core Function                                          | Typical Use                                                                    | Example                                                                                   |
|-------------|--------------------------------------------------------|--------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| **`system`** | Sets the AI's behavior mode, response rules, and operational constraints | - Define AI identity (e.g., command-line expert)<br>- Control output format (JSON/natural language)<br>- Set security permissions | `{"role": "system", "content": "You should determine the command to execute from the user's perspective. The current system is Linux"}` |
| **`user`**   | Passes the user's actual request or conversation input | - Single-turn direct questions (e.g., "view files")<br>- Multi-turn follow-ups (e.g., "What are the specific steps?") | `{"role": "user", "content": "Check if there is a main.go file in the current directory"}` |
| **`assistant`** | Stores the AI's historical replies for context continuity | - Provides coherent answers in multi-turn conversations<br>- Developers can modify historical replies to correct errors | `{"role": "assistant", "content": "Execute command: ls main.go"}` |

---

### Additional Notes:
1. **Role Interaction Rules**
   - **Message order**: The last message must be from the `user` role to trigger a response, and each request needs the complete context.
   - **Dynamic adjustment**: Inserting new `system` messages can switch the AI's identity (e.g., from translation assistant to data analyst).

2. **Performance Optimization**
   - `system` content should be concise (under 200 tokens) to avoid redundancy affecting model processing efficiency.

3. **Extended Applications**
   - Combined with `function calling`, you can use `system` to set function calling rules, `user` to trigger requests, and `assistant` to return structured parameters for executing external operations (such as calling APIs).

In [ ]:
# System role
completion = client.chat.completions.create(
    model="moonshotai/Kimi-K2.6",
    messages=[
        {"role": "system", "content": "You should determine the command to execute from the user's perspective. Only return the best option. The current system is Linux."},
        {"role": "user", "content": "How to check the operating system type"},
    ]
)
print(completion.choices[0].message.content)

## Multi-turn Conversation

Here is a summary of the core differences and characteristics between single-turn and multi-turn conversations:

---

### **Single-turn Conversation**
**Definition**: The user interacts with the AI only once. The AI generates a response based solely on the current input without retaining any conversation history.
**Characteristics**:
1. **No context dependency**: Each request is processed independently, suitable for simple Q&A or one-time tasks (e.g., generating articles, translating short sentences).
2. **Simple implementation**: No need to manage conversation history; API calls only require `system` instructions and `user` input.
3. **Low resource consumption**: Suitable for short texts (<500 tokens) with fast response times.

**Example scenarios**:
- Generating an article outline: User inputs a topic, AI directly outputs the complete structure.
- Translating a single sentence: User provides text to translate, AI returns the target language result.

**Code example** (OpenAI):
```python
messages = [
    {"role": "system", "content": "You are a translation assistant"},
    {"role": "user", "content": "Translate 'Hello' to Chinese"}
]
```

---

### **Multi-turn Conversation**
**Definition**: The user interacts with the AI multiple times, and the AI must maintain context for coherent dialogue.
**Characteristics**:
1. **Context management**: Pass conversation history through the `messages` array (containing `user` and `assistant` roles) so the AI understands the current conversation state.
2. **Dynamic adjustment**: New `system` instructions can be inserted or historical messages modified to flexibly control AI behavior.
3. **Complex scenario support**: Suitable for tasks requiring multi-step interaction (e.g., customer service, code debugging).

**Implementation challenges**:
- **Token limits**: Models have input length caps (e.g., DeepSeek supports 128K tokens); optimize by truncating history or generating summaries.
- **Performance optimization**: Streaming calls (returning chunks incrementally) improve long-text interaction experience but require handling data concatenation logic.

**Example scenarios**:
- Customer service: User asks about order status multiple times, AI answers progressively based on history.
- Programming assistant: User describes requirements step by step, AI generates and corrects code snippets.

**Code example** (context truncation strategy):
```python
# Keep only last 5 conversation pairs to control token count
messages = chat_history[-10:]
```

---

### **Technical Comparison**
| **Dimension**       | Single-turn                    | Multi-turn                       |
|---------------------|-------------------------------|----------------------------------|
| **Context dependency** | None                       | Strongly depends on history      |
| **Use cases**       | Simple tasks (translation, summarization) | Complex interaction (customer service, multi-step reasoning) |
| **Implementation complexity** | Low                  | High (requires context and token management) |
| **API call mode**   | Synchronous (returns complete response) | Supports streaming (returns chunks) |

---

### **Selection Guidelines**
- **Prefer single-turn**: When the task is simple and no historical reference is needed (e.g., generating a random recipe).
- **Choose multi-turn**: When coherent interaction and dynamic AI behavior adjustment are needed (e.g., tutoring, technical troubleshooting).
- **Use real-time API**: OpenAI's real-time API can automatically cache context, reducing management complexity.


In [ ]:
chat_history = []  # Save complete conversation history

# First round of conversation
user_input_1 = "What is the tallest mountain in the world?"
chat_history.append({"role": "user", "content": user_input_1})

response_1 = client.chat.completions.create(
    model="moonshotai/Kimi-K2.6",
    messages=chat_history
)
chat_history

In [ ]:
# Add assistant reply to history
ai_response_1 = response_1.choices[0].message.content
chat_history.append({"role": "assistant", "content": ai_response_1})  # Save AI reply

chat_history

In [ ]:
# Second round (context-dependent)
user_input_2 = "What about the second tallest?"
chat_history.append({"role": "user", "content": user_input_2})

response_2 = client.chat.completions.create(
    model="moonshotai/Kimi-K2.6",
    messages=chat_history[-10:]  # Truncation: keep only last 10 messages to control token usage
)

ai_response_2 = response_2.choices[0].message.content

# Add assistant reply to history
chat_history.append({"role": "assistant", "content": ai_response_2})

chat_history

In [ ]:
# Print complete conversation history
for msg in chat_history:
    print(f"{msg['role'].capitalize()}: {msg['content']}")

## IO Solutions Invitation Code

HGE53IOF

Free DeepSeek full version! Invite friends to register and use the platform. Both parties can receive up to 145 CNY in vouchers, covering 36.25 million tokens for free with R1 and V3 models! Entry: https://api.intelligence.io.solutions/api/v1  Invitation code: HGE53IOF

<img src='img/huoshan2.png' width='640px' />